<a href="https://www.kaggle.com/code/avikdas567/arc-agi-2-grid-transform-solver-with-beam-search?scriptVersionId=306627842" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import json
import numpy as np
from collections import Counter

DATA_PATH = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"

with open(f"{DATA_PATH}/arc-agi_test_challenges.json") as f:
    tasks = json.load(f)


# UTILS

def to_np(x): return np.array(x)
def to_list(x): return np.array(x).tolist()

def pixel_score(a,b):
    if a.shape != b.shape: return 0
    return np.mean(a==b)

def shape_score(a,b):
    return 1.0 if a.shape == b.shape else 0.0

def combined_score(a,b):
    return 0.7*pixel_score(a,b) + 0.3*shape_score(a,b)


# OPS

def identity(x): return x
def rot90(x): return np.rot90(x,1)
def rot180(x): return np.rot90(x,2)
def rot270(x): return np.rot90(x,3)
def flip_h(x): return np.fliplr(x)
def flip_v(x): return np.flipud(x)

def crop(x):
    coords = np.argwhere(x!=0)
    if len(coords)==0: return x
    x1,y1 = coords.min(0)
    x2,y2 = coords.max(0)
    return x[x1:x2+1, y1:y2+1]

def mirror_h(x):
    return np.concatenate([x, np.fliplr(x)], axis=1)

def mirror_v(x):
    return np.concatenate([x, np.flipud(x)], axis=0)

OPS = [identity, rot90, rot180, rot270, flip_h, flip_v, crop, mirror_h, mirror_v]


# COLOR MAP

def get_color_map(inp, out):
    if inp.shape != out.shape: return None
    mapping = {}
    for i in range(inp.shape[0]):
        for j in range(inp.shape[1]):
            a,b = inp[i,j], out[i,j]
            if a in mapping and mapping[a]!=b:
                return None
            mapping[a]=b
    return mapping

def apply_map(x, mapping):
    res = np.copy(x)
    for k,v in mapping.items():
        res[x==k]=v
    return res


# OBJECT CENTERING

def center_object(x):
    x = np.array(x)
    coords = np.argwhere(x!=0)
    if len(coords)==0:
        return x

    h,w = x.shape
    obj = x[coords[:,0], coords[:,1]]

    new = np.zeros_like(x)
    cx, cy = h//2, w//2

    for (i,j), val in zip(coords, obj):
        dx = i - coords[:,0].mean()
        dy = j - coords[:,1].mean()

        ni = int(cx + dx)
        nj = int(cy + dy)

        if 0<=ni<h and 0<=nj<w:
            new[ni,nj] = val

    return new


# PROGRAM CLASS

class Program:
    def __init__(self, ops):
        self.ops = ops

    def __call__(self, x):
        for op in self.ops:
            x = op(x)
        return x


# BEAM SEARCH

def beam_search(task, beam_width=6, depth=3):

    beam = [Program([identity])]

    for _ in range(depth):
        candidates = []

        for prog in beam:
            for op in OPS:

                new_prog = Program(prog.ops + [op])
                total_score = 0

                for p in task["train"]:
                    inp = to_np(p["input"])
                    out = to_np(p["output"])

                    try:
                        pred = new_prog(inp)
                    except:
                        total_score = -1
                        break

                    total_score += combined_score(pred, out)

                candidates.append((total_score, new_prog))

        candidates.sort(key=lambda x: -x[0])
        beam = [p for _,p in candidates[:beam_width]]

    return beam


# SOLVER

def solve(task):

    programs = beam_search(task)
    cmap = None

    # try color map
    for p in task["train"]:
        m = get_color_map(to_np(p["input"]), to_np(p["output"]))
        if m is None:
            cmap = None
            break
        cmap = m

    results=[]

    for tcase in task["test"]:
        inp = to_np(tcase["input"])

        preds=[]

        # program predictions
        for prog in programs:
            try:
                preds.append(prog(inp))
            except:
                pass

        # object centering
        preds.append(center_object(inp))

        # color map
        if cmap is not None:
            preds.append(apply_map(inp, cmap))

        # fallback
        if len(preds)==0:
            preds=[inp]

        # remove duplicates
        unique=[]
        for p in preds:
            if not any(np.array_equal(p,q) for q in unique):
                unique.append(p)

        # pick best 2 diverse
        p1 = unique[0]
        p2 = unique[1] if len(unique)>1 else unique[0]

        results.append({
            "attempt_1": to_list(p1),
            "attempt_2": to_list(p2)
        })

    return results


# RUN

submission = {}

for tid,task in tasks.items():
    submission[tid] = solve(task)

with open("submission.json","w") as f:
    json.dump(submission,f)

print("Submission Created!")

Submission Created!
